# CosyVoice 2 — Production Indian English Podcast

**Apache 2.0** — enterprise/commercial use OK.

This notebook combines everything we learned:
- Professional **LIMMITS'24** reference voices (studio quality, 2 speakers)
- Zero-shot voice cloning (no training needed)
- Per-line speed control for energy variation
- Phonetic pronunciation fixes (A I, Preeya, start up)
- Natural filler words between speakers (hmm, right, absolutely)
- Smart pauses (longer after questions, natural between hosts)
- Audio post-processing (EQ + compression) for podcast feel
- Works on both Colab and RunPod

| Step | What | Time |
|------|------|------|
| 1 | Setup + install | ~10 min |
| 2 | Download CosyVoice 2 model | ~5 min |
| 3 | Download LIMMITS'24 references | ~20 min (first time only) |
| 4 | Load model + pick references | ~5 min |
| 5 | Generate production podcast | ~10 min |
| 6 | Audio post-processing | ~2 min |

## Step 1: Setup

In [ ]:
import os, sys, shutil, glob

# --- Detect environment ---
IS_COLAB = os.path.exists('/content') and 'COLAB_RELEASE_TAG' in os.environ
BASE = '/content' if IS_COLAB else '/workspace'
print(f"Environment: {'Colab' if IS_COLAB else 'RunPod/Other'}")
print(f"Base: {BASE}")

# --- GPU ---
import torch
assert torch.cuda.is_available(), "No GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- HuggingFace Login ---
try:
    if IS_COLAB:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

if not os.environ.get('HF_TOKEN'):
    !pip install -q huggingface_hub
    from huggingface_hub import login
    login()

# --- Mount Drive / set backup ---
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = '/content/drive/MyDrive/cosyvoice2_production'
else:
    BACKUP_DIR = f'{BASE}/backup_cosyvoice2'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backup: {BACKUP_DIR}")

In [ ]:
# Install CosyVoice and all dependencies
!apt-get -qq install -y sox libsox-dev ffmpeg > /dev/null 2>&1

COSYVOICE_DIR = f'{BASE}/CosyVoice'
os.chdir(BASE)
if not os.path.exists(f'{COSYVOICE_DIR}/.git'):
    !rm -rf {COSYVOICE_DIR}
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSYVOICE_DIR}
    !cd {COSYVOICE_DIR} && git submodule update --init --recursive

# Install all required packages (learned from previous errors)
!pip install -q hyperpyyaml modelscope onnxruntime soundfile librosa \
    openai-whisper conformer diffsptk inflect pydub einops omegaconf \
    huggingface_hub datasets torchaudio num2words \
    'transformers>=4.45,<4.50' sentencepiece \
    diffusers accelerate safetensors tokenizers \
    hydra-core lightning pytorch-lightning \
    gdown matplotlib wget pyworld \
    2>&1 | tail -3

print("\nInstall complete!")

## Step 2: Download CosyVoice 2 Model

In [ ]:
sys.path.insert(0, COSYVOICE_DIR)
sys.path.insert(0, f'{COSYVOICE_DIR}/third_party/Matcha-TTS')

MODEL_DIR = f'{COSYVOICE_DIR}/pretrained_models/CosyVoice2-0.5B'
if not os.path.exists(f'{MODEL_DIR}/llm.pt'):
    from huggingface_hub import snapshot_download
    snapshot_download('FunAudioLLM/CosyVoice2-0.5B', local_dir=MODEL_DIR)
    print("Model downloaded!")
else:
    print("Model already cached.")

## Step 3: Download LIMMITS'24 Professional Reference Voices

**Studio-quality** Indian English (CC-BY 4.0) — 2 professional voice actors.
This is the KEY to good quality. We only need a few clips, not the full 80 hours.

In [ ]:
# Download just 2 sample files (not the full tar.gz)
LIMMITS_DIR = f'{BASE}/limmits_refs'
os.makedirs(f'{LIMMITS_DIR}/male', exist_ok=True)
os.makedirs(f'{LIMMITS_DIR}/female', exist_ok=True)

# Check if we already have LIMMITS data from previous runs
existing_m = glob.glob(f'{BASE}/data_limmits/limmits_raw/English_M/wav/*.wav')
existing_f = glob.glob(f'{BASE}/data_limmits/limmits_raw/English_F/wav/*.wav')

if existing_m and existing_f:
    print(f"Found existing LIMMITS data: {len(existing_m)} male, {len(existing_f)} female clips")
    MALE_WAV_DIR = f'{BASE}/data_limmits/limmits_raw/English_M/wav'
    MALE_TXT_DIR = f'{BASE}/data_limmits/limmits_raw/English_M/txt'
    FEMALE_WAV_DIR = f'{BASE}/data_limmits/limmits_raw/English_F/wav'
    FEMALE_TXT_DIR = f'{BASE}/data_limmits/limmits_raw/English_F/txt'
else:
    print("Downloading LIMMITS'24 (full — ~21 GB)...")
    print("This is one-time. The tar.gz files are needed for reference extraction.")

    from huggingface_hub import hf_hub_download
    import tarfile

    LIMMITS_RAW = f'{BASE}/data_limmits/limmits_raw'
    os.makedirs(LIMMITS_RAW, exist_ok=True)

    for gender, filename in [("female", "English_F.tar.gz"), ("male", "English_M.tar.gz")]:
        tar_path = os.path.join(LIMMITS_RAW, filename)
        if not os.path.exists(tar_path):
            print(f"  Downloading {filename}...")
            hf_hub_download(
                repo_id="iAkashPaul/limmits-2024",
                filename=filename,
                repo_type="dataset",
                local_dir=LIMMITS_RAW,
            )

        # Extract
        marker = os.path.join(LIMMITS_RAW, f'.{gender}_extracted')
        if not os.path.exists(marker):
            print(f"  Extracting {filename}...")
            with tarfile.open(tar_path, "r:gz") as tar:
                tar.extractall(path=LIMMITS_RAW)
            open(marker, 'w').close()

    MALE_WAV_DIR = f'{LIMMITS_RAW}/English_M/wav'
    MALE_TXT_DIR = f'{LIMMITS_RAW}/English_M/txt'
    FEMALE_WAV_DIR = f'{LIMMITS_RAW}/English_F/wav'
    FEMALE_TXT_DIR = f'{LIMMITS_RAW}/English_F/txt'

print(f"\nMale wav dir: {MALE_WAV_DIR}")
print(f"Female wav dir: {FEMALE_WAV_DIR}")

## Step 4: Load Model + Pick Reference Voices

In [ ]:
# Load CosyVoice 2
os.chdir(COSYVOICE_DIR)
from cosyvoice.cli.cosyvoice import CosyVoice2
import torchaudio

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B')
sr = cosyvoice.sample_rate
print(f"Model loaded! Sample rate: {sr}")

In [ ]:
# Browse LIMMITS clips — pick the cleanest ones
import IPython.display as ipd
import soundfile as sf
import numpy as np

def get_transcript(wav_path, txt_dir):
    stem = os.path.splitext(os.path.basename(wav_path))[0]
    txt_path = os.path.join(txt_dir, f'{stem}.txt')
    if os.path.exists(txt_path):
        return open(txt_path, 'r', encoding='utf-8').read().strip()
    return ""

def find_clean_clip(wav_dir, txt_dir, min_dur=6, max_dur=12):
    """Find clips in good duration range with energy."""
    wavs = sorted(glob.glob(f'{wav_dir}/*.wav'))
    candidates = []
    for wav in wavs[:200]:  # check first 200
        try:
            data, file_sr = sf.read(wav)
            dur = len(data) / file_sr
            if min_dur <= dur <= max_dur:
                rms = np.sqrt(np.mean(data**2))
                if rms > 0.02:  # not silent
                    candidates.append(wav)
                    if len(candidates) >= 5:
                        break
        except:
            continue
    return candidates

male_candidates = find_clean_clip(MALE_WAV_DIR, MALE_TXT_DIR)
female_candidates = find_clean_clip(FEMALE_WAV_DIR, FEMALE_TXT_DIR)

print("=" * 60)
print("  MALE candidates (LIMMITS professional voice)")
print("=" * 60)
for i, wav in enumerate(male_candidates):
    data, file_sr = sf.read(wav)
    dur = len(data) / file_sr
    txt = get_transcript(wav, MALE_TXT_DIR)
    print(f"\n#{i} ({dur:.1f}s): {txt[:70]}...")
    ipd.display(ipd.Audio(wav))

print("\n" + "=" * 60)
print("  FEMALE candidates (LIMMITS professional voice)")
print("=" * 60)
for i, wav in enumerate(female_candidates):
    data, file_sr = sf.read(wav)
    dur = len(data) / file_sr
    txt = get_transcript(wav, FEMALE_TXT_DIR)
    print(f"\n#{i} ({dur:.1f}s): {txt[:70]}...")
    ipd.display(ipd.Audio(wav))

In [ ]:
# === PICK YOUR REFERENCES ===
# Based on what sounded best above, set the index here
MALE_IDX = 0     # Change based on what you liked
FEMALE_IDX = 0

male_ref = male_candidates[MALE_IDX]
female_ref = female_candidates[FEMALE_IDX]
male_ref_text = get_transcript(male_ref, MALE_TXT_DIR)
female_ref_text = get_transcript(female_ref, FEMALE_TXT_DIR)

print(f"Male ref: {os.path.basename(male_ref)}")
print(f"  Text: {male_ref_text[:80]}")
ipd.display(ipd.Audio(male_ref))

print(f"\nFemale ref: {os.path.basename(female_ref)}")
print(f"  Text: {female_ref_text[:80]}")
ipd.display(ipd.Audio(female_ref))

## Step 5: Generate Production Podcast

All the tuning from previous iterations:
- Per-line speed control
- Phonetic pronunciation fixes
- Natural fillers
- Smart pauses

In [ ]:
import time

# ============================================================
# PODCAST SCRIPT — (speaker, text, speed)
# Speed: 1.0=calm, 1.1=normal, 1.2=energetic, 1.3=very energetic
# ============================================================
PODCAST_SCRIPT = [
    ("female", "Welcome to A I India, the podcast where we explore how artificial intelligence is transforming our country. I am Preeya.", 1.25),
    ("male", "And I am Arjun. Today we are talking about something really exciting. The rise of Indian A I start ups.", 1.25),
    ("female", "That is right, Arjun. India now has over three hundred A I start ups, and that number is growing every single month.", 1.2),
    ("male", "What I find really interesting is that many of these companies are solving uniquely Indian problems. Like agriculture, health care in rural areas, and education.", 1.15),
    ("female", "Absolutely. Take for example an A I system that can detect crop diseases just by looking at a photo taken on a farmer's mobile phone.", 1.2),
    ("male", "And in health care, A I models are now screening for conditions like diabetic retinopathy and tuberculosis in areas where there are very few doctors available.", 1.15),
    ("female", "The language barrier is another big challenge that A I is helping with. India has twenty two official languages and hundreds of dialects.", 1.15),
    ("male", "Exactly. And that is precisely why building speech technology like text to speech systems in Indian languages is so important.", 1.2),
    ("female", "Speaking of which, the progress in Indian language A I has been remarkable. Models can now understand and generate speech in Hindi, Tamil, Bengali, and many more.", 1.15),
    ("male", "The government has also been supportive with initiatives to build open source data sets for Indian languages. This is a game changer.", 1.2),
    ("female", "So what do you think is next for A I in India, Arjun?", 1.25),
    ("male", "I believe we will see A I becoming a part of everyday life. From voice assistants that truly understand Indian accents, to A I tutors that teach children in their mother tongue.", 1.15),
    ("female", "That is a beautiful vision. And it all starts with building the right foundation, the right data, the right models, and the right talent.", 1.1),
    ("male", "Could not agree more. India has the talent, and now we are building the tools.", 1.25),
    ("female", "That is all for today's episode of A I India. Thank you for listening, and we will see you next week.", 1.2),
    ("male", "Goodbye everyone, and keep innovating!", 1.3),
]

# Pause settings (seconds)
PAUSE_SAME_SPEAKER = 0.3
PAUSE_SWITCH_SPEAKER = 0.5
PAUSE_AFTER_QUESTION = 0.8
PAUSE_AFTER_FILLER = 0.4

# Filler words
FILLERS_MALE = ["hmm", "right", "yes", "absolutely"]
FILLERS_FEMALE = ["hmm", "yes", "right", "indeed"]
ADD_FILLERS = True

def generate_filler(word, ref_wav, ref_txt):
    """Generate a filler word — spoken clearly."""
    try:
        chunks = []
        for result in cosyvoice.inference_zero_shot(
            word + ".", ref_txt, ref_wav, stream=False, speed=1.0
        ):
            chunks.append(result['tts_speech'].squeeze().numpy())
        if chunks:
            audio = np.concatenate(chunks)
            audio = audio[:int(sr * 1.2)]
            # Trim trailing silence
            threshold = 0.01
            for end_idx in range(len(audio) - 1, 0, -1):
                if abs(audio[end_idx]) > threshold:
                    break
            audio = audio[:end_idx + int(sr * 0.05)]
            fade = min(int(sr * 0.05), len(audio))
            audio[-fade:] *= np.linspace(1, 0, fade)
            return audio * 0.7
    except Exception:
        pass
    return np.zeros(int(sr * 0.3))

# Generate
OUTPUT_DIR = f'{BASE}/outputs/production_podcast'
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_segments = []
prev_speaker = None
filler_idx_male = 0
filler_idx_female = 0

print("Generating production podcast with LIMMITS references...\n")
start = time.time()

for i, (speaker, text, speed) in enumerate(PODCAST_SCRIPT):
    name = "Priya" if speaker == "female" else "Arjun"
    ref_wav = female_ref if speaker == "female" else male_ref
    ref_txt = female_ref_text if speaker == "female" else male_ref_text

    # Pauses and fillers
    if prev_speaker is not None:
        prev_text = PODCAST_SCRIPT[i-1][1]
        if prev_text.endswith('?'):
            all_segments.append(np.zeros(int(sr * PAUSE_AFTER_QUESTION)))
        elif speaker != prev_speaker:
            all_segments.append(np.zeros(int(sr * PAUSE_SWITCH_SPEAKER)))

            if ADD_FILLERS and i > 1 \
               and not text.startswith("Welcome") \
               and not text.startswith("That is all") \
               and not text.startswith("Goodbye"):
                if speaker == "male":
                    fw = FILLERS_MALE[filler_idx_male % len(FILLERS_MALE)]
                    filler_idx_male += 1
                else:
                    fw = FILLERS_FEMALE[filler_idx_female % len(FILLERS_FEMALE)]
                    filler_idx_female += 1
                print(f"          ({name}: \"{fw}\")")
                filler = generate_filler(fw, ref_wav, ref_txt)
                all_segments.append(filler)
                all_segments.append(np.zeros(int(sr * PAUSE_AFTER_FILLER)))
        else:
            all_segments.append(np.zeros(int(sr * PAUSE_SAME_SPEAKER)))

    # Generate main line
    gen_start = time.time()
    chunks = []
    for result in cosyvoice.inference_zero_shot(text, ref_txt, ref_wav, stream=False, speed=speed):
        chunks.append(result['tts_speech'].squeeze().numpy())

    audio = np.concatenate(chunks) if chunks else np.zeros(sr)

    # Fade in/out
    fade_len = min(int(sr * 0.02), len(audio) // 4)
    audio[:fade_len] *= np.linspace(0, 1, fade_len)
    audio[-fade_len:] *= np.linspace(1, 0, fade_len)

    gen_time = time.time() - gen_start
    duration = len(audio) / sr
    print(f"  [{name:5s}] {duration:.1f}s (spd:{speed:.2f}) | {text[:50]}...")

    sf.write(f'{OUTPUT_DIR}/line_{i:02d}_{speaker}.wav', audio, sr)
    all_segments.append(audio)
    prev_speaker = speaker

full_audio = np.concatenate(all_segments)
RAW_PODCAST = f'{OUTPUT_DIR}/podcast_raw.wav'
sf.write(RAW_PODCAST, full_audio, sr)

total_time = time.time() - start
total_dur = len(full_audio) / sr
print(f"\nRaw podcast: {total_dur:.0f}s ({total_dur/60:.1f} min) | Gen: {total_time:.0f}s")

## Step 6: Audio Post-Processing (EQ + Compression)

Makes it sound like a real podcast instead of raw TTS output.

In [ ]:
# Apply podcast-style audio processing
FINAL_PODCAST = f'{OUTPUT_DIR}/podcast_final.wav'

# ffmpeg filter chain:
# - highpass: remove rumble below 80Hz
# - lowpass: remove harsh frequencies above 12000Hz
# - compand: dynamic range compression (podcast-standard)
# - loudnorm: normalize to -16 LUFS (podcast standard)
# - equalizer: slight presence boost around 3kHz for clarity

ffmpeg_filter = (
    "highpass=f=80,"
    "lowpass=f=12000,"
    "equalizer=f=3000:t=q:w=1:g=2,"
    "compand=attacks=0.05:decays=0.3:points=-80/-80|-50/-15|-20/-5|0/-3:gain=3,"
    "loudnorm=I=-16:TP=-1.5:LRA=11"
)

!ffmpeg -y -i {RAW_PODCAST} -af "{ffmpeg_filter}" -ar 44100 {FINAL_PODCAST} 2>&1 | tail -3

print(f"\nFinal podcast: {FINAL_PODCAST}")
print("\nRAW (before processing):")
ipd.display(ipd.Audio(RAW_PODCAST))
print("\nFINAL (with EQ + compression + loudness):")
ipd.display(ipd.Audio(FINAL_PODCAST))

# Backup to Drive
shutil.copy2(FINAL_PODCAST, os.path.join(BACKUP_DIR, 'podcast_final.wav'))
shutil.copy2(RAW_PODCAST, os.path.join(BACKUP_DIR, 'podcast_raw.wav'))
print(f"\nBacked up to: {BACKUP_DIR}")

## Try Different References or Tuning

If you want to experiment — change MALE_IDX/FEMALE_IDX in Step 4 or edit PODCAST_SCRIPT speeds in Step 5, then re-run those cells.

In [ ]:
# Quick pronunciation test
test_text = "Welcome to A I India. I am Preeya. Today we discuss Indian start ups."
test_speed = 1.2

print("[MALE]")
for result in cosyvoice.inference_zero_shot(test_text, male_ref_text, male_ref, stream=False, speed=test_speed):
    a = result['tts_speech'].squeeze().numpy()
ipd.display(ipd.Audio(a, rate=sr))

print("[FEMALE]")
for result in cosyvoice.inference_zero_shot(test_text, female_ref_text, female_ref, stream=False, speed=test_speed):
    a = result['tts_speech'].squeeze().numpy()
ipd.display(ipd.Audio(a, rate=sr))